# Convert to Pseudobulk by count sums

In this notebook, we sum UMI counts, and compute means after LogNormalize in Seurat to generate pseudobulk values for each cell type per sample.

### Output structure

This notebook generates 3 matrices, as well as sample x cell type metadata:

`agg_mat`: Raw count aggregates generated as the sum of UMIs per gene within each sample x cell type group  
`mean_mat`: Mean of normalized aggregates generated by performing `LogNormalize()` in Seurat prior to computing means  
`detect_mat`: Counts of gene detection frequency within each group, which can be used for feature selection  

Sample metadata (`sample_meta`) has the following columns:

`cohort.cohortGuid`: Cohort ID  
`subject.ageAtFirstDraw`: Subject Age at first on-study blood draw  
`subject.biologicalSex`: Subject's biological sex (Female or Male)  
`subject.birthYear`: Subject's year of birth  
`subject.bmi`: Subject's BMI, rounded to integer  
`subject.cmv`: Subject's CMV status (Negative or Positive)  
`subject.ethnicity`: Subject's self-reported ethnicity  
`subject.race`: Subject's self-reported race  
`subject.subjectGuid`: Unique Subject ID  
`sample.drawYear`: Sample collection year (YYYY)  
`sample.sampleKitGuid`: Unique Sample ID  
`sample.subjectAgeAtDraw`: Age of subject at time of sample draw  
`sample.visitName`: Name of sample collection visit  
`specimen.specimenGuid`: Unique ID of specific aliquot used to generate data  
`batch_id`: Batch ID for quality control  
`pool_id`: Pool ID for quality control (usually 2 sample pools per batch)  
`AIFI_L1`: Broad cell type label  
`AIFI_L2`: Intermediate resolution cell type label  
`AIFI_L3`: High resolution cell type label  
`n_cells`: Number of cells used to generate pseudobulk counts  
`barcodes`: Unique identifier for this pseudobulk population, with the structure `{subject.subjectGuid}_{sample.visitName}_{AIFI_L3}`.
- `barcodes` matches the column names of the matrices

## Load packages

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }

quiet_library(dplyr)
quiet_library(hise)
quiet_library(H5weaver)
quiet_library(purrr)
quiet_library(furrr)
quiet_library(Seurat)
quiet_library(tidyr)

Warning message:
“package ‘data.table’ was built under R version 4.4.3”
Warning message:
“package ‘Matrix’ was built under R version 4.4.3”
Warning message:
“package ‘rhdf5’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.2”
Warning message:
“package ‘future’ was built under R version 4.4.3”
Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Warning message:
“package ‘sp’ was built under R version 4.4.2”


In [2]:
plan(multicore, workers = 16)

In [3]:
exclude <- c("^LINC","^MT","^RP")

In [4]:
if(!dir.exists("output")) {
    dir.create("output")
}
if(!dir.exists("pseudobulk_l3")) {
    dir.create("pseudobulk_l3")
}

In [5]:
out_files <- c()

## Helper functions

This function formats cell types for use in filenames

In [6]:
format_cell_type <- function(cell_type) {
    cell_type <- gsub("\\+", "pos", cell_type)
    cell_type <- gsub("-", "neg", cell_type)
    cell_type <- gsub(" ", "-", cell_type)
    cell_type
}

This function assists in reading cell metadata data directly from .h5ad files into R

In [7]:
read_h5ad_cell_meta <- function(h5ad_file) 
{
    h5ad_contents <- H5weaver::h5ls(h5ad_file)
    obs_locs <- h5ad_contents$full_name[h5ad_contents$group == "/obs"]
    obs_locs <- obs_locs[!obs_locs %in% c("/obs/__categories", "/obs/_index")]
    obs_locs <- obs_locs[!grepl("Unnamed", obs_locs)]

    h5ad <- H5Fopen(h5ad_file)

    obs_list <- lapply(obs_locs, function(loc) {h5read(h5ad, loc)})

    obs_list <- lapply(
        obs_list,
        function(obs) {
            if(length(obs) == 2) {
                vals <- vector(length = length(obs$codes))
                vals[obs$codes >= 0] <- as.vector(obs$categories)[as.vector(obs$codes + 1)]
                vals[obs$codes == -1] <- NA
            } else {
                vals <- as.vector(obs)
            }

            vals
        }
    )

    obs_list <- lapply(obs_list, as.vector)
    names(obs_list) <- sub(".+/", "", obs_locs)

    H5Fclose(h5ad)
    as.data.frame(obs_list)
}

This function converts from .h5ad expression values to pseudobulk

In [8]:
sum_list_to_matrix <- function(sum_list, col_names, row_names) {
    mat <- matrix(unlist(sum_list), ncol = length(sum_list))
    colnames(mat) <- col_names
    rownames(mat) <- row_names
    mat
}

In [9]:
sample_h5ad_to_l3_pseudobulk <- function(sample_h5ad_file, exclude = NULL) {
    # Read cell metadata
    meta <- read_h5ad_cell_meta(sample_h5ad_file)
    subject <- meta$subject.subjectGuid[1]
    visit <- meta$sample.visitName[1]
    
    sample_prefix <- paste0(subject, "_", gsub(" ", "-",visit), "_")
    
    # Format cell type so they can be used as names
    meta <- meta %>%
      mutate(format_AIFI_L3 = format_cell_type(AIFI_L3))
    
    # Read counts
    mat <- read_h5ad_dgCMatrix(sample_h5ad_file, feature_names = "_index")
    
    # Filter genes if needed
    genes <- rownames(mat)
    if(!is.null(exclude)) {
        keep_genes <- genes[!grepl("^RP|^MT-|^LINC",genes)]
    } else {
        keep_genes <- genes
    }
    
    # Filter for selected genes and ensure order matches metadata
    mat <- mat[keep_genes, meta$barcodes]

    # Split metadata and matrices by AIFI_L3 type
    split_meta <- split(meta, meta$format_AIFI_L3)
    split_mats <- map(
        split_meta,
        function(meta) { 
            # Transpose so each gene is a column
            t(mat[, meta$barcodes, drop = FALSE])
        }
    )
    
    # Sum counts and detection for each gene
    agg_sums <- map(split_mats, function(mat) { colSums(mat) })
    detect_sums <- map(split_mats, function(mat) { diff(mat@p) })

    # Normalize with Seurat
    so <- CreateSeuratObject(
        counts = mat,
        meta.data = meta
    )
    so <- NormalizeData(
        so, 
        normalization.method = "LogNormalize", 
        scale.factor = 1e4,
        verbose = FALSE
    )

    # Extract and transpose normalized data
    norm_mats <- map(
        split_meta,
        function(meta) {
            mat <- so[["RNA"]]@layers$data
            colnames(mat) <- so@meta.data$barcodes
            mat <- mat[,meta$barcodes]
            t(mat)
        }
    )

    # Mean of normalized counts
    norm_means <- map(
        norm_mats,
        function(mat, n) {
            colSums(mat) / nrow(mat)
        })
    
    # Assemble matrices from sums
    aggregate_names <- paste0(sample_prefix, names(split_meta))

    agg_mat <- sum_list_to_matrix(agg_sums, aggregate_names, keep_genes)
    mean_mat <- sum_list_to_matrix(norm_means, aggregate_names, keep_genes)
    detect_mat <- sum_list_to_matrix(detect_sums, aggregate_names, keep_genes)
    
    # Generate aggregate metadata
    type_columns <- c("AIFI_L1", "AIFI_L2", "AIFI_L3")

    meta <- meta %>%
      select(cohort.cohortGuid,
             starts_with("subject"),
             starts_with("sample"),
             starts_with("specimen"),
             "batch_id", "pool_id",
             one_of(type_columns)) %>%
      group_by(AIFI_L3) %>%
      mutate(n_cells = n()) %>%
      ungroup() %>%
      unique() %>%
      mutate(barcodes = paste0(sample_prefix, format_cell_type(AIFI_L3))) %>%
      arrange(AIFI_L3)

    #print(head(meta$barcodes))
    agg_mat <- agg_mat[,meta$barcodes]
    mean_mat <- mean_mat[,meta$barcodes]
    detect_mat <- detect_mat[,meta$barcodes]
    
    list(
        agg_mat = agg_mat,
        mean_mat = mean_mat,
        detect_mat = detect_mat,
        sample_meta = meta
    )
}

In [10]:
select_pseudobulk_samples <- function(pb_data, ...) {
    pb_data$sample_meta <- pb_data$sample_meta %>%
      filter( ... )

    pb_data$agg_mat <- pb_data$agg_mat[, pb_data$sample_meta$barcodes]
    pb_data$mean_mat <- pb_data$mean_mat[, pb_data$sample_meta$barcodes]
    pb_data$detect_mat <- pb_data$detect_mat[, pb_data$sample_meta$barcodes]

    pb_data
}

## Retrieve files to process in HISE

We'll retrieve our clean, non-normalized .h5ad datasets for each sample from HISE

In [11]:
search_id <- "polonium-tin-curium"

In [12]:
ps_files <- listFilesInProjectStores(list("cohorts"))
ps_files <- map(
    ps_files$files, 
    function(l) {
        l <- l[c("id", "name")]
        as.data.frame(l)
    }) %>%
  list_rbind()

In [13]:
tar_files <- ps_files %>%
  filter(grepl(search_id, name)) %>%
  filter(grepl(".tar$", name))

## Retrieve and unpack sample files

In [14]:
if(!dir.exists("sample_h5ad")) {
    walk(tar_files$id,
         function(uuid) {
            if(!dir.exists(paste0("cache/", uuid))) {
                hise_res <- cacheFiles(list(uuid))
            }
            
            tar_file <- hise_res
            untar_call <- paste("tar -xf", tar_file)
            system(untar_call)
        }
    )
}

## Convert to pseudobulk for each sample

Now, we'll iterate through each file and apply our pseudobulk function to each in parallel.

In [15]:
possibly_convert <- possibly(sample_h5ad_to_l3_pseudobulk, quiet = FALSE)

In [16]:
sample_h5ads <- list.files("sample_h5ad", full.names = TRUE)
results_list <- future_map(sample_h5ads, possibly_convert)

In [17]:
length(results_list)

[1] 868

In [18]:
sum(is.null(results_list))

[1] 0

### Restructure and combine results

In [24]:
result_names <- names(results_list[[1]])

pb_data <- map(
    result_names, 
    function(result_name) {
        map(results_list, result_name)
    }
)

names(pb_data) <- result_names

In [25]:
pb_data$agg_mat <- do.call(cbind, pb_data$agg_mat)
pb_data$mean_mat <- do.call(cbind, pb_data$mean_mat)
pb_data$detect_mat <- do.call(cbind, pb_data$detect_mat)
pb_data$sample_meta <- do.call(rbind, pb_data$sample_meta)

### Substitute drawYear for drawDate

In [27]:
pb_data$sample_meta <- pb_data$sample_meta %>%
  mutate(sample.drawDate = sub("-.+", "", sample.drawDate)) %>%
  rename(sample.drawYear = sample.drawDate)

## Save to .rds for later use in R

In [29]:
out_rds <- paste0("output/diha_AIFI_L3_pseudobulk_list_", Sys.Date(), ".rds")
saveRDS(pb_data, out_rds)

In [30]:
out_files <- c(out_files, out_rds)

## Save per cell type

In [31]:
cell_types <- unique(pb_data$sample_meta$AIFI_L3)
walk(
    cell_types,
    function(cell_type) {
        type_data <- pb_data %>%
            select_pseudobulk_samples(AIFI_L3 == cell_type)

        out_type <- format_cell_type(cell_type)
        type_file <- paste0("pseudobulk_l3/diha_", out_type, "_pseudobulk.rds")
        saveRDS(type_data, type_file)
    }
)

In [32]:
out_tar <- paste0("output/diha_AIFI_L3_pseudobulk_per_type_", Sys.Date(), ".tar")
tar_call <- paste("tar -cf", out_tar, "pseudobulk_l3/*.rds")
system(tar_call)

In [33]:
out_files <- c(out_files, out_tar)

## Save to .csv for flexible downstream usage

In [34]:
agg_csv <- paste0("output/diha_AIFI_L3_pseudobulk_agg_", Sys.Date(), ".csv")
agg_df <- as.data.frame(pb_data$agg_mat) %>%
  mutate(gene = rownames(pb_data$agg_mat)) %>%
  select(gene, everything())
fwrite(agg_df, agg_csv)

In [35]:
out_files <- c(out_files, agg_csv)

In [36]:
mean_csv <- paste0("output/diha_AIFI_L3_pseudobulk_mean_", Sys.Date(), ".csv")
mean_df <- as.data.frame(pb_data$mean_mat) %>%
  mutate(gene = rownames(pb_data$mean_mat)) %>%
  select(gene, everything())
fwrite(mean_df, mean_csv)

In [37]:
out_files <- c(out_files, mean_csv)

In [38]:
detect_csv <- paste0("output/diha_AIFI_L3_pseudobulk_detect_", Sys.Date(), ".csv")
detect_df <- as.data.frame(pb_data$detect_mat) %>%
  mutate(gene = rownames(pb_data$detect_mat)) %>%
  select(gene, everything())
fwrite(detect_df, detect_csv)

In [39]:
out_files <- c(out_files, detect_csv)

In [40]:
meta_csv <- paste0("output/diha_AIFI_L3_pseudobulk_meta_", Sys.Date(), ".csv")
fwrite(pb_data$sample_meta, meta_csv)

In [41]:
out_files <- c(out_files, meta_csv)

## Upload results to HISE

In [42]:
study_space_uuid <- "de025812-5e73-4b3c-9c3b-6d0eac412f2a"
title <- paste("DIHA scRNA L3 Pseudobulk", Sys.Date())

In [43]:
search_id <- ids::proquint(n_words = 3)
search_id

[1] "vuhuj-gafir-ruhub"

In [44]:
in_list <- as.list(tar_files$id)
in_list

[[1]]
[1] "04666e28-8443-4a51-8670-f409a7b5afe5"

[[2]]
[1] "ae2996c3-eab5-4d61-a997-084351727413"

[[3]]
[1] "b8f48340-ec96-4ed9-bad1-23fcb1a64e70"

[[4]]
[1] "11d754d9-0323-400b-8c47-8b9193d254d6"

[[5]]
[1] "6c6c9bbb-ac49-42f1-9e4f-f6a00766f331"

[[6]]
[1] "712082ed-2fe2-4121-9f89-7f732b4a58a7"

[[7]]
[1] "e1fe73c4-44d1-4092-ba72-72c5efe657d1"

[[8]]
[1] "dd3c4973-439f-4987-ac52-12cd86b31021"

In [45]:
out_list <- as.list(out_files)
out_list

[[1]]
[1] "output/diha_AIFI_L3_pseudobulk_list_2025-06-10.rds"

[[2]]
[1] "output/diha_AIFI_L3_pseudobulk_per_type_2025-06-10.tar"

[[3]]
[1] "output/diha_AIFI_L3_pseudobulk_agg_2025-06-10.csv"

[[4]]
[1] "output/diha_AIFI_L3_pseudobulk_mean_2025-06-10.csv"

[[5]]
[1] "output/diha_AIFI_L3_pseudobulk_detect_2025-06-10.csv"

[[6]]
[1] "output/diha_AIFI_L3_pseudobulk_meta_2025-06-10.csv"

In [46]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    destination = search_id
)

[1] "Retrying..."


checking if conda env can compile...



[1] "/home/workspace/environment/minimal"


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "27accd2a-0f34-4f97-897d-4342457d75a1"

$ProcessId
[1] "655a81d2-9cc7-48b9-93b8-e9bf4cfa318c"

$WorkflowId
[1] "c273ee19-d719-47ac-a7cd-48acca244d83"

$FileIds
$FileIds[[1]]
[1] "cc114d60-90fc-446d-8367-f91b105e9d32"

$FileIds[[2]]
[1] "5dea1ee8-965f-41cc-a478-56df35cf46b5"

$FileIds[[3]]
[1] "78f8c7e0-cd2a-48c2-9618-74f151ebf386"

$FileIds[[4]]
[1] "0be7945d-08f2-4d9c-a2e3-c0e2b8413be4"

$FileIds[[5]]
[1] "447bfca2-9793-4afd-84e8-726938cd29a3"

$FileIds[[6]]
[1] "a2d2c904-3a77-4f46-8faf-f06806724d52"

In [47]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/minimal/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] tidyr_1.3.1        Seurat_5.1.0       SeuratObject_5.1.0 sp_2.2-0          
 [5] furrr_0.3.1        future_1.58.0      purrr_1.0.4        H5weaver_1.3.0    
 [9] rhdf5_2.50.0       Matrix_1.7-3       data.table_1.17.4  hise_2.16.0       
[13] dplyr_1.1.4       

loaded via a namespace (and not attached):
  [1] RCol